# Fil rouge `xyt_gps` avec le test set anonymisé

Ce notebook est le parcours à montrer en premier. Il part du dossier `anonymous-test-set-gps`, prépare les tables de mobilité, contrôle les utilisateurs, calcule les indicateurs et écrit un export propre.

Le jeu de données utilisé ici est anonymisé et limité à 12 utilisateurs. Il ne contient pas les questionnaires réels ; la table `sociodemographics` est synthétique. Pour tester le package hors dépôt, demander le dossier à `contact@situee.ch`, puis définir `XYT_GPS_DEMO_DATA_DIR`.

## 1. Préparer l'environnement

Le notebook fonctionne dans deux situations : depuis le dépôt de développement, ou après installation du package. Les sorties locales sont écrites dans `examples/_outputs/fil-rouge-clean-export`.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

# En développement local, on importe le code source du dépôt.
# Après publication, `pip install xyt-gps` suffira et ce bloc n'aura aucun effet.
for root in (Path.cwd(), *Path.cwd().parents):
    candidates = [root / "package-xyt-gps" / "src", root / "src"]
    for candidate in candidates:
        if (candidate / "xyt_gps").exists() and str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

import xyt_gps as xyt


def find_demo_data_dir() -> Path:
    """Localiser le dossier anonymous-test-set-gps."""

    env_path = os.environ.get("XYT_GPS_DEMO_DATA_DIR")
    candidates = []
    if env_path:
        candidates.append(Path(env_path).expanduser())
    for root in (Path.cwd(), *Path.cwd().parents):
        candidates.extend([
            root / "Data" / "Output" / "anonymous-test-set-gps",
            root / "anonymous-test-set-gps",
        ])
    for candidate in candidates:
        if (candidate / "storyline.parquet").exists():
            return candidate.resolve()
    raise FileNotFoundError(
        "Jeu de démonstration introuvable. "
        "Définir XYT_GPS_DEMO_DATA_DIR ou demander le dossier anonymous-test-set-gps à contact@situee.ch."
    )


def example_output_dir(name: str) -> Path:
    """Créer un dossier de sortie local pour les exports et figures du notebook."""

    for root in (Path.cwd(), *Path.cwd().parents):
        if (root / "package-xyt-gps" / "examples").exists():
            path = root / "package-xyt-gps" / "examples" / "_outputs" / name
            path.mkdir(parents=True, exist_ok=True)
            return path
        if (root / "examples").exists() and (root / "src" / "xyt_gps").exists():
            path = root / "examples" / "_outputs" / name
            path.mkdir(parents=True, exist_ok=True)
            return path
    path = Path.cwd() / "xyt_gps_example_outputs" / name
    path.mkdir(parents=True, exist_ok=True)
    return path


DATA_DIR = find_demo_data_dir()
OUTPUT_DIR = example_output_dir("fil-rouge-clean-export")

pd.Series({"data_dir": DATA_DIR.name, "output_dir": f"examples/_outputs/{OUTPUT_DIR.name}"}, name="environnement")

## 2. Charger le test set

On charge les tables GPS anonymisées et la petite table sociodémographique synthétique. Le manifeste rappelle les hypothèses de génération du jeu de démonstration.

In [ ]:
tables = {
    "storyline": pd.read_parquet(DATA_DIR / "storyline.parquet"),
    "trips": pd.read_parquet(DATA_DIR / "trips.parquet"),
    "journeys": pd.read_parquet(DATA_DIR / "journeys.parquet"),
    "user_statistics": pd.read_parquet(DATA_DIR / "user_statistics.parquet"),
    "sociodemographics": pd.read_parquet(DATA_DIR / "sociodemographics.parquet"),
}
manifest = json.loads((DATA_DIR / "test_set_manifest.json").read_text(encoding="utf-8"))

volume = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in tables.items()
    ]
)

display(pd.Series(manifest, name="manifest"))
display(volume)

## 3. Définir le périmètre analytique

`ProjectConfig` rend visibles les phases, la période, le CRS de travail et les paramètres de transformation. Ici les phases sont des phases de démonstration ; les lignes hors période restent visibles sous `Other`, ce qui permet de diagnostiquer le périmètre plutôt que de le masquer.

In [ ]:
phases = (
    xyt.Phase("Phase1", "2025-04-14", "2025-05-04"),
    xyt.Phase("Phase2", "2025-05-05", "2025-06-01"),
    xyt.Phase("Phase3", "2025-06-02", "2025-06-15"),
)

config = xyt.ProjectConfig(
    experiment_name="anonymous-test-set-gps",
    motiontag_project_name="test-project-gps",
    start_expe="2025-04-14",
    end_expe="2025-06-15",
    phases=phases,
    reference_year=2025,
)

raw = xyt.RawGpsData(
    storyline=tables["storyline"],
    trips=tables["trips"],
    journeys=tables["journeys"],
    user_statistics=tables["user_statistics"],
)

schema_report = xyt.check_raw_import_columns(
    raw.storyline,
    raw.user_statistics,
    trips=raw.trips,
    journeys=raw.journeys,
    raise_on_error=False,
)

schema_report.groupby(["table", "status"], dropna=False).size().rename("columns").reset_index()

## 4. Transformer les exports GPS en tables de mobilité

`prepare_mobility_dataset()` construit les staypoints, les legs, les liens avec trips/journeys, les statistiques utilisateur et les champs qualité. C'est le chemin court recommandé pour un analyste qui veut démarrer vite.

In [ ]:
dataset = xyt.prepare_mobility_dataset(
    raw,
    config,
    sociodemo=tables["sociodemographics"],
    resample_missing_days=True,
    clean_leg_geometries=True,
    add_length_outlier_flags=True,
    add_signal_quality_flags=True,
)

mobility_tables = xyt.mobility_dataset_tables(dataset)
summary = pd.DataFrame(
    [
        {
            "table": name,
            "rows": len(table),
            "columns": len(table.columns),
            "users": table["user_id"].nunique() if "user_id" in table.columns else None,
        }
        for name, table in mobility_tables.items()
    ]
)

display(summary)
display(dataset.legs[["user_id", "started_at", "finished_at", "mode_niv1", "phase", "length"]].head())

## 5. Contrôler les utilisateurs et calculer les indicateurs

Le test set est volontairement petit. On garde ici des filtres lisibles et peu restrictifs pour montrer la mécanique ; dans un projet réel, les seuils de qualité doivent être explicités dans la configuration.

In [ ]:
selection = xyt.build_user_selection_table(
    dataset.user_stats,
    require_tracking_quality=False,
    exclude_bad_signal_users=False,
)

indicators = xyt.compute_mobility_indicators(dataset, config=config)
indicator_summary = xyt.population_indicator_summary(indicators)

selection_cols = ["user_id", "active_days_count", "tracking_completeness", "bad_signal_user", "analysis_user_ok", "analysis_user_reason"]
display(selection[[col for col in selection_cols if col in selection.columns]].head(12))
display(indicator_summary.head(12))

## 6. Écrire un export propre

`export_clean_dataset()` regroupe les tables structurées, les indicateurs, la sélection utilisateurs et les tables additionnelles. C'est la sortie conseillée pour alimenter des analyses ou un dashboard.

In [ ]:
clean_export_dir = OUTPUT_DIR / "clean-export"
manifest = xyt.export_clean_dataset(
    dataset,
    indicators,
    clean_export_dir,
    selection_table=selection,
    extra_tables={"sociodemographics": tables["sociodemographics"]},
    separate_format_dirs=True,
    overwrite=True,
)

display(manifest.groupby(["section", "format"], dropna=False).agg(tables=("table", "nunique"), rows=("rows", "sum")).reset_index())
print(f"Export écrit dans : examples/_outputs/{OUTPUT_DIR.name}/{clean_export_dir.name}")

## 7. Points à retenir

- `RawGpsData` contient les tables brutes ou déjà landed.
- `ProjectConfig` garde les choix méthodologiques explicites.
- `MobilityDataset` est l'objet de travail inspectable.
- `IndicatorResult` regroupe les indicateurs personne-jour, personne-phase et population.
- Le manifeste d'export est le meilleur point de contrôle pour vérifier ce qui a été écrit.